# PredNet — Prediction Error & Lucas-Kanade Flow Analysis
Upload `fpsi136_410000.pth` and `MonoWheel.mp4` before running.

In [ ]:
!pip install hickle --quiet

In [ ]:
import os, math, glob
import numpy as np
import torch
import torch.nn as nn
from torch.nn import Parameter, functional as F
from torch.nn.modules.utils import _pair
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import subprocess, cv2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding=1, dilation=1, groups=1, bias=True):
        super().__init__()
        if in_channels % groups != 0:
            raise ValueError('in_channels must be divisible by groups')
        if out_channels % groups != 0:
            raise ValueError('out_channels must be divisible by groups')
        kernel_size = _pair(kernel_size)
        stride      = _pair(stride)
        padding     = _pair(padding)
        dilation    = _pair(dilation)
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size  = kernel_size
        self.stride       = stride
        self.padding      = padding
        self.padding_h    = tuple(
            k // 2 for k, s, p, d in zip(kernel_size, stride, padding, dilation))
        self.dilation = dilation
        self.groups   = groups
        self.weight_ih = Parameter(torch.Tensor(4 * out_channels, in_channels  // groups, *kernel_size))
        self.weight_hh = Parameter(torch.Tensor(4 * out_channels, out_channels // groups, *kernel_size))
        self.weight_ch = Parameter(torch.Tensor(3 * out_channels, out_channels // groups, *kernel_size))
        if bias:
            self.bias_ih = Parameter(torch.Tensor(4 * out_channels))
            self.bias_hh = Parameter(torch.Tensor(4 * out_channels))
            self.bias_ch = Parameter(torch.Tensor(3 * out_channels))
        else:
            self.register_parameter('bias_ih', None)
            self.register_parameter('bias_hh', None)
            self.register_parameter('bias_ch', None)
        self.register_buffer('wc_blank', torch.zeros(1, 1, 1, 1))
        self.reset_parameters()

    def reset_parameters(self):
        n = 4 * self.in_channels
        for k in self.kernel_size: n *= k
        stdv = 1. / math.sqrt(n)
        self.weight_ih.data.uniform_(-stdv, stdv)
        self.weight_hh.data.uniform_(-stdv, stdv)
        self.weight_ch.data.uniform_(-stdv, stdv)
        if self.bias_ih is not None:
            self.bias_ih.data.uniform_(-stdv, stdv)
            self.bias_hh.data.uniform_(-stdv, stdv)
            self.bias_ch.data.uniform_(-stdv, stdv)

    def forward(self, input, hx):
        h_0, c_0 = hx
        wx   = F.conv2d(input, self.weight_ih, self.bias_ih,
                        self.stride, self.padding, self.dilation, self.groups)
        wh   = F.conv2d(h_0,   self.weight_hh, self.bias_hh,
                        self.stride, self.padding_h, self.dilation, self.groups)
        wc   = F.conv2d(c_0,   self.weight_ch, self.bias_ch,
                        self.stride, self.padding_h, self.dilation, self.groups)
        wxhc = wx + wh + torch.cat((
            wc[:, :2 * self.out_channels],
            self.wc_blank.expand(wc.size(0), wc.size(1) // 3, wc.size(2), wc.size(3)),
            wc[:, 2 * self.out_channels:]
        ), dim=1)
        i   = torch.sigmoid(wxhc[:, :self.out_channels])
        f   = torch.sigmoid(wxhc[:,  self.out_channels:2 * self.out_channels])
        g   = torch.tanh(   wxhc[:, 2 * self.out_channels:3 * self.out_channels])
        o   = torch.sigmoid(wxhc[:, 3 * self.out_channels:])
        c_1 = f * c_0 + i * g
        h_1 = o * torch.tanh(c_1)
        return h_1, (h_1, c_1)

In [ ]:
class SatLU(nn.Module):
    def __init__(self, lower=0, upper=255):
        super().__init__()
        self.lower = lower
        self.upper = upper
    def forward(self, x):
        return F.hardtanh(x, self.lower, self.upper)

class PredNet(nn.Module):
    def __init__(self, R_channels, A_channels, output_mode='error'):
        super().__init__()
        assert len(R_channels) == len(A_channels)
        if output_mode not in ('prediction', 'error'):
            raise ValueError(f'Invalid output_mode "{output_mode}"')
        self.r_channels  = R_channels
        self.a_channels  = A_channels
        self.n_layers    = len(R_channels)
        self.output_mode = output_mode
        r_channels_above = R_channels[1:] + (0,)
        for l in range(self.n_layers):
            setattr(self, f'cell{l}',
                ConvLSTMCell(2 * A_channels[l] + r_channels_above[l], R_channels[l], kernel_size=3))
        for l in range(self.n_layers):
            conv = nn.Sequential(
                nn.Conv2d(R_channels[l], A_channels[l], kernel_size=3, padding=1), nn.ReLU())
            if l == 0: conv.add_module('satlu', SatLU())
            setattr(self, f'conv{l}', conv)
        self.maxpool  = nn.MaxPool2d(kernel_size=2, stride=2)
        self.upsample = nn.Upsample(scale_factor=2)
        for l in range(self.n_layers - 1):
            setattr(self, f'update_A{l}', nn.Sequential(
                nn.Conv2d(2 * A_channels[l], A_channels[l + 1], kernel_size=3, padding=1),
                self.maxpool))
        self.reset_parameters()

    def reset_parameters(self):
        for l in range(self.n_layers):
            getattr(self, f'cell{l}').reset_parameters()

    def forward(self, input):
        batch_size, time_steps = input.size(0), input.size(1)
        H, W = input.size(-2), input.size(-1)
        dev  = input.device
        E_seq, R_seq, H_seq = [], [], [None] * self.n_layers
        for l in range(self.n_layers):
            ds = 2 ** l
            E_seq.append(torch.zeros(batch_size, 2 * self.a_channels[l], H // ds, W // ds, device=dev))
            R_seq.append(torch.zeros(batch_size,     self.r_channels[l], H // ds, W // ds, device=dev))
        total_error, frame_prediction = [], None
        for t in range(time_steps):
            A = input[:, t].float()
            for l in reversed(range(self.n_layers)):
                cell = getattr(self, f'cell{l}')
                hx   = H_seq[l] if H_seq[l] is not None else (R_seq[l], R_seq[l])
                lstm_input = E_seq[l] if l == self.n_layers - 1 \
                             else torch.cat([E_seq[l], self.upsample(R_seq[l + 1])], dim=1)
                R_seq[l], H_seq[l] = cell(lstm_input, hx)
            for l in range(self.n_layers):
                A_hat = getattr(self, f'conv{l}')(R_seq[l])
                if l == 0: frame_prediction = A_hat
                pos = F.relu(A_hat - A)
                neg = F.relu(A - A_hat)
                E_seq[l] = torch.cat([pos, neg], dim=1)
                if l < self.n_layers - 1:
                    A = getattr(self, f'update_A{l}')(E_seq[l])
            if self.output_mode == 'error':
                total_error.append(torch.cat(
                    [e.flatten(1).mean(1, keepdim=True) for e in E_seq], dim=1))
        if self.output_mode == 'error':
            return torch.stack(total_error, dim=2)
        return frame_prediction

In [ ]:
WEIGHTS_PATH = 'fpsi136_410000.pth'
A_channels   = (3, 48, 96, 192)
R_channels   = (3, 48, 96, 192)
model = PredNet(R_channels, A_channels, output_mode='prediction')
ckpt  = torch.load(WEIGHTS_PATH, map_location=device)
sd    = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
model.load_state_dict(sd)
model = model.to(device)
model.eval()
print(f'Loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
FRAMES_DIR = 'monowheel_frames'
os.makedirs(FRAMES_DIR, exist_ok=True)
subprocess.run([
    'ffmpeg', '-y', '-i', 'MonoWheel.mp4',
    '-vf', 'scale=160:120', '-q:v', '2',
    f'{FRAMES_DIR}/frame_%04d.jpg'
], check=True, capture_output=True)
frame_paths = sorted(glob.glob(f'{FRAMES_DIR}/frame_*.jpg'))
print(f'Extracted {len(frame_paths)} frames')

In [ ]:
def load_frame(path):
    img = np.asarray(Image.open(path)).transpose(2, 0, 1).astype(np.float32) / 255.0
    return img

def tensor_to_uint8(t):
    """(3,H,W) or (1,3,H,W) float -> HxWx3 uint8"""
    return (t.squeeze().permute(1,2,0).clamp(0,1).numpy() * 255).astype(np.uint8)

frames       = np.stack([load_frame(p) for p in frame_paths])
input_tensor = torch.from_numpy(frames).unsqueeze(0).to(device)
print(f'Tensor shape: {input_tensor.shape}')

In [ ]:
CHUNK     = 30
T         = input_tensor.shape[1]
all_preds = []
all_mae   = []

with torch.no_grad():
    for start in range(0, T - 1, CHUNK):
        end   = min(start + CHUNK + 1, T)
        chunk = input_tensor[:, start:end]
        pred  = model(chunk)
        all_preds.append(pred.cpu())
        mae   = torch.mean(torch.abs(pred.cpu() - chunk[:, -1].cpu())).item()
        all_mae.append(mae)

print(f'Inference done. Mean MAE: {np.mean(all_mae):.4f}')

In [ ]:
# ── Wheel mask: detect the wheel region via thresholding ─────────────────
# Uses the first GT frame — the wheel is the bright non-black region.
# Returns a boolean mask (H, W) where True = inside wheel.

def make_wheel_mask(frame_uint8, threshold=15):
    """frame_uint8: HxWx3. Returns HxW bool mask of non-background region."""
    gray   = cv2.cvtColor(frame_uint8, cv2.COLOR_RGB2GRAY)
    _, mask = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
    # Clean up with morphological closing
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask.astype(bool)

first_gt   = tensor_to_uint8(input_tensor[:, 0].cpu())
wheel_mask = make_wheel_mask(first_gt)
print(f'Wheel mask: {wheel_mask.sum()} px inside / {(~wheel_mask).sum()} px outside')

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(first_gt);           axes[0].set_title('GT frame 0');    axes[0].axis('off')
axes[1].imshow(wheel_mask, cmap='gray'); axes[1].set_title('Wheel mask'); axes[1].axis('off')
overlay = first_gt.copy()
overlay[~wheel_mask] = (overlay[~wheel_mask] * 0.25).astype(np.uint8)
axes[2].imshow(overlay);            axes[2].set_title('Masked GT');      axes[2].axis('off')
plt.tight_layout()
plt.savefig('wheel_mask.png', dpi=150)
plt.show()

In [ ]:
# ── Prediction error maps + inside/outside decomposition ─────────────────
sample_chunks = list(range(0, len(all_preds), max(1, len(all_preds) // 6)))[:6]

inside_errors  = []
outside_errors = []
frame_labels   = []

fig, axes = plt.subplots(len(sample_chunks), 3,
                         figsize=(9, 3 * len(sample_chunks)))
fig.suptitle('Per-pixel absolute prediction error', fontsize=13, y=1.01)

for row, idx in enumerate(sample_chunks):
    frame_idx = min(idx * CHUNK + CHUNK, T - 1)
    frame_labels.append(frame_idx)

    gt_img   = tensor_to_uint8(input_tensor[:, frame_idx].cpu())
    pred_img = tensor_to_uint8(all_preds[idx])

    # Per-pixel absolute error (mean over channels), normalised to [0,1]
    err_map  = np.mean(np.abs(gt_img.astype(float) - pred_img.astype(float)), axis=2) / 255.0

    # Inside / outside stats
    inside_errors.append(err_map[wheel_mask].mean())
    outside_errors.append(err_map[~wheel_mask].mean())

    # GT with mask contour overlaid
    contour_img = gt_img.copy()
    contours, _ = cv2.findContours(
        wheel_mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(contour_img, contours, -1, (255, 80, 0), 1)

    axes[row, 0].imshow(contour_img)
    axes[row, 0].set_title(f'GT f{frame_idx}', fontsize=9)
    axes[row, 0].axis('off')

    im = axes[row, 1].imshow(err_map, cmap='hot', vmin=0, vmax=0.3)
    axes[row, 1].set_title(f'Error map (MAE {all_mae[idx]:.3f})', fontsize=9)
    axes[row, 1].axis('off')
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046, pad=0.04)

    # Error map with inside/outside split shown as contour
    split_img = plt.cm.hot(err_map / 0.3)[:, :, :3]
    # Dim the outside region slightly for visual separation
    split_img[~wheel_mask] = split_img[~wheel_mask] * 0.5
    split_img = (split_img * 255).astype(np.uint8)
    cv2.drawContours(split_img, contours, -1, (0, 200, 255), 1)
    axes[row, 2].imshow(split_img)
    axes[row, 2].set_title(
        f'Inside {inside_errors[-1]:.3f}  /  Outside {outside_errors[-1]:.3f}', fontsize=9)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig('prednet_error_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prednet_error_maps.png')

In [ ]:
# ── Inside vs outside error summary bar chart ─────────────────────────────
x      = np.arange(len(sample_chunks))
width  = 0.35
labels = [f'f{l}' for l in frame_labels]

fig, ax = plt.subplots(figsize=(10, 4))
bars_in  = ax.bar(x - width/2, inside_errors,  width, label='Inside wheel',  color='#E24B4A')
bars_out = ax.bar(x + width/2, outside_errors, width, label='Outside wheel', color='#378ADD')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Mean absolute error')
ax.set_title('Prediction error: inside vs outside wheel region')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Annotate ratio
for i, (inn, out) in enumerate(zip(inside_errors, outside_errors)):
    ratio = inn / out if out > 0 else float('inf')
    ax.text(x[i], max(inn, out) + 0.002, f'{ratio:.1f}x', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('prednet_error_inside_outside.png', dpi=150)
plt.show()

print(f'Mean inside error:  {np.mean(inside_errors):.4f}')
print(f'Mean outside error: {np.mean(outside_errors):.4f}')
print(f'Ratio (inside/outside): {np.mean(inside_errors)/np.mean(outside_errors):.2f}x')

In [ ]:
# ── Lucas-Kanade sparse optical flow: GT->GT and GT->Pred ─────────────────
# Tracks Shi-Tomasi good features from frame N to frame N+1,
# separately on real frames and on predicted frames.

LK_PARAMS = dict(
    winSize=(15, 15),
    maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01)
)
FEATURE_PARAMS = dict(maxCorners=80, qualityLevel=0.02, minDistance=5, blockSize=7)


def draw_lk_flow(img_bgr, p0, p1, status, color):
    """Draw LK tracks on a copy of img_bgr. Returns RGB."""
    out = img_bgr.copy()
    for i, (new, old) in enumerate(zip(p1, p0)):
        if status[i]:
            a, b = new.ravel().astype(int)
            c, d = old.ravel().astype(int)
            cv2.arrowedLine(out, (c, d), (a, b), color, 1, tipLength=0.4)
            cv2.circle(out, (c, d), 2, color, -1)
    return cv2.cvtColor(out, cv2.COLOR_BGR2RGB)


fig, axes = plt.subplots(len(sample_chunks), 2,
                         figsize=(8, 3.5 * len(sample_chunks)))
fig.suptitle('Lucas-Kanade sparse flow  —  Left: GT→GT  |  Right: GT→Pred', fontsize=12, y=1.01)

lk_mag_gt_list   = []
lk_mag_pred_list = []

for row, idx in enumerate(sample_chunks):
    frame_idx = min(idx * CHUNK + CHUNK, T - 1)
    prev_idx  = max(frame_idx - 1, 0)

    gt_prev_rgb  = tensor_to_uint8(input_tensor[:, prev_idx].cpu())
    gt_curr_rgb  = tensor_to_uint8(input_tensor[:, frame_idx].cpu())
    pred_curr_rgb = tensor_to_uint8(all_preds[idx])

    gt_prev_gray  = cv2.cvtColor(gt_prev_rgb,   cv2.COLOR_RGB2GRAY)
    gt_curr_gray  = cv2.cvtColor(gt_curr_rgb,   cv2.COLOR_RGB2GRAY)
    pred_curr_gray = cv2.cvtColor(pred_curr_rgb, cv2.COLOR_RGB2GRAY)

    # Detect features in the previous GT frame
    p0 = cv2.goodFeaturesToTrack(gt_prev_gray, mask=None, **FEATURE_PARAMS)

    if p0 is None:
        axes[row, 0].axis('off')
        axes[row, 1].axis('off')
        continue

    # GT->GT flow
    p1_gt,   st_gt,   _ = cv2.calcOpticalFlowPyrLK(gt_prev_gray,   gt_curr_gray,   p0, None, **LK_PARAMS)
    # GT->Pred flow
    p1_pred, st_pred, _ = cv2.calcOpticalFlowPyrLK(gt_prev_gray, pred_curr_gray, p0, None, **LK_PARAMS)

    # Magnitude stats (tracked points only)
    def tracked_mag(p_src, p_dst, status):
        vecs = (p_dst[status.ravel() == 1] - p_src[status.ravel() == 1]).reshape(-1, 2)
        return np.linalg.norm(vecs, axis=1).mean() if len(vecs) else 0.0

    lk_mag_gt_list.append(tracked_mag(p0, p1_gt,   st_gt))
    lk_mag_pred_list.append(tracked_mag(p0, p1_pred, st_pred))

    gt_prev_bgr = cv2.cvtColor(gt_prev_rgb, cv2.COLOR_RGB2BGR)

    axes[row, 0].imshow(draw_lk_flow(gt_prev_bgr, p0, p1_gt,   st_gt,   (0, 255, 100)))
    axes[row, 0].set_title(f'GT f{frame_idx}  mag={lk_mag_gt_list[-1]:.2f}px', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(draw_lk_flow(gt_prev_bgr, p0, p1_pred, st_pred, (255, 80, 0)))
    axes[row, 1].set_title(f'Pred f{frame_idx}  mag={lk_mag_pred_list[-1]:.2f}px', fontsize=9)
    axes[row, 1].axis('off')

plt.tight_layout()
plt.savefig('prednet_lk_flow.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: prednet_lk_flow.png')

print(f'\nMean LK magnitude  GT→GT:   {np.mean(lk_mag_gt_list):.3f} px')
print(f'Mean LK magnitude  GT→Pred: {np.mean(lk_mag_pred_list):.3f} px')

In [ ]:
# ── Summary stats printout ────────────────────────────────────────────────
print('=' * 50)
print('PREDICTION ERROR SUMMARY')
print('=' * 50)
print(f'Overall mean MAE:          {np.mean(all_mae):.4f}')
print(f'Mean error inside wheel:   {np.mean(inside_errors):.4f}')
print(f'Mean error outside wheel:  {np.mean(outside_errors):.4f}')
print(f'Inside/outside ratio:      {np.mean(inside_errors)/np.mean(outside_errors):.2f}x')
print()
print('LUCAS-KANADE FLOW SUMMARY')
print('=' * 50)
print(f'Mean LK magnitude GT→GT:   {np.mean(lk_mag_gt_list):.3f} px  (actual motion)')
print(f'Mean LK magnitude GT→Pred: {np.mean(lk_mag_pred_list):.3f} px  (predicted motion)')
print(f'Pred/GT magnitude ratio:   {np.mean(lk_mag_pred_list)/np.mean(lk_mag_gt_list):.2f}x')